# Beyond the Stopwatch (V6 Championship Edition): 4-Position 10 Hz Dual-Head Attention & GM Draft Surplus EV
**NFL Big Data Bowl 2027 ($100,000 Prize Pool) | Reproducible Official-Data, 328 Colab L4 GPU Models & GM Surplus EV Notebook**

This notebook reproduces our complete zero-leakage V4–V6 analytics stack directly on the official NFL Big Data Bowl 2027 dataset (510 rookies, 431,094 optical Combine 10 Hz frames, 314,197 plays, and 22,654,290 regular-season NFL game tracking frames from 2023–2025):
- **Overall V4–V6 OOF R²:** `+0.2562` (`95% CI [+0.1954, +0.3150]`) | **Temporal LOYO R²:** `+0.2286` (`q_FDR = 0.0264` vs V2 `+0.2049`)
- **Position-Specific Breakout (`DL_EDGE`):** OOF **`R² = +0.6892`**, **Temporal LOYO `R² = +0.7045`** (`Partial R² = +0.3566`, `q_FDR = 0.0012`)
- **Incremental Signal Beyond Draft Capital + Stopwatch:** **`Partial R² = +0.1121`** (`q_FDR = 0.0012`)
- **GM Draft-Room Surplus EV (`draft_ev.py`):** CBA-calibrated 4-year rookie contract surplus (`$M`), bust risk reduction (`%`), and real historical comps.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

KAGGLE_DIR = Path("/kaggle/input/nfl-big-data-bowl-2027")
LOCAL_DIR = Path("/Volumes/Extra/datasets/kaggle-nfl2027/nfl-big-data-bowl-2027")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else LOCAL_DIR

NB_ROOT = Path.cwd()
SRC_CANDIDATES = [NB_ROOT / "../src", NB_ROOT / "src", NB_ROOT / "code/nfl-bdb2027/src"]
SRC_DIR = next((p.resolve() for p in SRC_CANDIDATES if p.exists()), SRC_CANDIDATES[0].resolve())
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nfl_bdb2027.config import ARTIFACTS_DIR, MATERIALS_DIR

print(f"Official data directory: {DATA_DIR}")
print(f"Package src directory:   {SRC_DIR}")

In [ ]:
from nfl_bdb2027.data import DatasetPaths, audit_dataset_inventory, load_combine_results
from nfl_bdb2027.kinematics import aggregate_player_kinematics, extract_rep_level_kinematics
from nfl_bdb2027.outcomes import build_outcomes_table

paths = DatasetPaths.from_dir(DATA_DIR)
inventory = audit_dataset_inventory(paths)
print(f"Prospects: {inventory['total_prospects']} | Combine Player Frames: {inventory['combine_tracking_player_rows']:,}")
print(f"Play Rows: {inventory['player_play_rows']:,} | NFL Game Tracking Frames: {inventory['game_tracking_total_rows']:,}")
print(f"40-yd Opt-Outs: {inventory['forty_opt_out_count']} | 3-Cone Opt-Outs: {inventory['three_cone_opt_out_count']} | Skill Coverage: {inventory['combine_tracking_coverage']['players_with_skill_tracking']}/510")

In [ ]:
# Extract 10 Hz optical Combine kinematics & exposure-normalized NFL outcomes (100% Polars vectorized)
rep_df = extract_rep_level_kinematics(paths)
cr_pl = load_combine_results(paths)
outcomes_df = build_outcomes_table(paths)
full_kin = aggregate_player_kinematics(rep_df, cr_pl, fit_nfl_ids=None).to_pandas()
master_df = outcomes_df.merge(full_kin, on="nfl_id", how="left")
print(f"Drill reps extracted: {rep_df.height:,} | Master prospect table: {master_df.shape}")

In [ ]:
# Audit V4-V6 4-Position Multi-Scale Dual-Head Attention GPU Results & Benjamini-Hochberg FDR Rigor
# (Trained on Colab NVIDIA L4 via scripts/train_v4_colab_gpu.py across 328 strictly zero-leakage nested models)
v4_metrics = json.loads((MATERIALS_DIR / "real_data_metrics_v4_gpu.json").read_text(encoding="utf-8"))
v4_saliency = json.loads((MATERIALS_DIR / "v4_saliency_profiles.json").read_text(encoding="utf-8"))

ov = v4_metrics["overall_v4_oof"]
print(
    f"Overall V4 OOF R2: {ov['cv_r2']:+.4f} [{ov['cv_r2_ci95'][0]:+.4f}, {ov['cv_r2_ci95'][1]:+.4f}] | "
    f"Temporal LOYO: {ov['temporal_loyo_r2']:+.4f} | Partial R2 vs Draft+SW: {ov['partial_r2_vs_draft_stopwatch']:+.4f} (q_FDR={ov['q_fdr_vs_draft_stopwatch']:.4f})"
)

pos_rows = []
for grp, pm in v4_metrics["positional_encoders"].items():
    pos_rows.append({
        "Position Group": grp,
        "n": pm["n_players"],
        "Stopwatch R2": round(pm["stopwatch_r2"], 4),
        "V2 Unified R2": round(pm["v2_unified_r2"], 4),
        "V4 OOF R2": round(pm["cv_r2"], 4),
        "Temporal LOYO R2": round(pm["temporal_loyo_r2"], 4),
        "Partial R2 vs SW": round(pm["partial_r2_vs_stopwatch"], 4),
        "q_FDR": round(pm["q_fdr_vs_stopwatch"], 4),
        "Peak 0.3s Window (s)": f"{pm['peak_saliency_window_sec'][0]:.2f}-{pm['peak_saliency_window_sec'][1]:.2f}",
    })
pos_df = pd.DataFrame(pos_rows)
print(pos_df.to_string(index=False))

In [ ]:
# Calibrate NFL GM Draft-Room 4-Year Rookie Surplus EV Engine on all 510 prospects + V4 GPU OOF scores
from nfl_bdb2027.draft_ev import DraftSurplusEVEngine

v4_npz_path = ARTIFACTS_DIR / "v4_positional_attn_gpu_oof.npz"
v4_oof_scores = {}
if v4_npz_path.exists():
    v4_npz = np.load(v4_npz_path, allow_pickle=True)
    v4_oof_scores = {
        int(v4_npz["nfl_ids"][int(orig_idx)]): np.asarray(v4_npz["v4_b1_strat_oof_feats"][row_i], dtype=np.float64)
        for row_i, orig_idx in enumerate(v4_npz["v4_b1_strat_sub_indices"])
        if np.all(np.isfinite(v4_npz["v4_b1_strat_oof_feats"][row_i]))
    }
ev_engine = DraftSurplusEVEngine.fit(master_df, v4_oof_scores=v4_oof_scores)
ev_bundle = ev_engine.export_ui_and_chart_bundle()

for grp, pk, oz in [("WR_TE", 177, 0.6), ("DL_EDGE", 54, 0.5), ("DB", 24, -1.4)]:
    ev = ev_engine.evaluate(grp, pk, oz)
    top_comp = ev["historical_comps"][0]
    print(
        f"{grp:7s} Pick #{pk:3d} (Rd {ev['draft_round']}, {oz:+.1f}σ) -> "
        f"Surplus EV {ev['expected_surplus_ev_m']:+.2f}M (Prod ${ev['expected_production_4yr_m']:.2f}M - Cost ${ev['rookie_contract_4yr_m']:.2f}M) | "
        f"Bust Delta {-ev['bust_risk_reduction_pct']:+.1f}% | Verdict: {ev['verdict']} | "
        f"#1 Comp: {top_comp['display_name']} (Realized {top_comp['realized_surplus_4yr_m']:+.1f}M)"
    )

In [ ]:
# Verify all 6 Economist/NYT pure-white figures (fig1-fig6) & V5/V6 Interactive Film-Room Studio artifacts
from nfl_bdb2027.charts import generate_all_figures
from nfl_bdb2027.filmroom_v5 import write_v5_interactive_filmroom
from nfl_bdb2027.report import write_kaggle_writeup_md

v2_metrics = json.loads((MATERIALS_DIR / "real_data_metrics_v2.json").read_text(encoding="utf-8"))
master_df["optical_alpha_z"] = ev_engine.prospects_df["optical_alpha_z"].to_numpy()
out_dir = MATERIALS_DIR if (MATERIALS_DIR.exists() and os.access(MATERIALS_DIR, os.W_OK)) else Path("./outputs")
out_dir.mkdir(parents=True, exist_ok=True)

fig_paths = generate_all_figures(
    paths=paths,
    results=v2_metrics["modeling_results"],
    master_df=master_df,
    figures_dir=out_dir / "figures",
    v4_metrics=v4_metrics,
    v4_saliency=v4_saliency,
    ev_bundle=ev_bundle,
)
md_path = write_kaggle_writeup_md(
    inventory=inventory,
    results=v2_metrics["modeling_results"],
    master_df=master_df,
    out_path=out_dir / "NFL_BDB2027_Kaggle_Writeup.md",
    v4_metrics=v4_metrics,
    v4_saliency=v4_saliency,
    ev_bundle=ev_bundle,
)
html_path = write_v5_interactive_filmroom(
    paths=paths,
    v4_metrics=v4_metrics,
    v4_saliency=v4_saliency,
    ev_bundle=ev_bundle,
    out_path=out_dir / "NFL_BDB2027_Interactive_FilmRoom.html",
)
print(f"Verified {len(fig_paths)} SVG figures (fig1-fig6), Writeup ({md_path.stat().st_size:,} B), FilmRoom ({html_path.stat().st_size:,} B)")